## Bootstrap (click Run All — no setup required)

Auto-installs packages and downloads Apollo 15/17 PDS data on first run. Subsequent runs are cached.

In [ ]:
# === Lunar-V2 bootstrap — safe to re-run =================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / 'pyproject.toml').is_file() and (_p / 'lunar' / '_bootstrap.py').is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError('Could not find Lunar-V2 repo root from ' + str(_here))

from lunar import _bootstrap as boot
boot.ensure_lunar(extra=('spiceypy', 'scipy'))
boot.ensure_apollo_hfe(mission='a15',
                       probes=('p1f1', 'p1f2', 'p1f3', 'p1f4',
                               'p2f1', 'p2f2', 'p2f3', 'p2f4'))
boot.ensure_apollo_hfe(mission='a17', probes=())
boot.ensure_spice_kernels()


# Apollo 15 & 17 HFE — Thermal Model Validation

**Goal.** Benchmark the Lunar-V2 1-D thermal solver against the Apollo Heat Flow Experiment (HFE) at Hadley-Apennine (A15, 26°N) and Taurus-Littrow (A17, 20°N). Two models are compared:

| Model | Conductivity K(T,z) | Calibration anchor |
|---|---|---|
| **Hayne 2017** | H-parameter exponential + χ(T/350)³ (App. A) | Diviner surface brightness T |
| **Discrete 3-layer** | Sharp layers + same χ(T/350)³ | Apollo subsurface heat-flow gradient |

Both share the same Crank-Nicolson solver, geothermal flux lower BC, and Hayne 2017 c_p(T) polynomial.

**Key figures produced:**
- **Fig 1** — Hayne (2017) thermophysical properties vs depth
- **Fig 2** — Mean T(z) profile: Hayne + Discrete vs Apollo HFE (both sites)
- **Fig 3** — Per-sensor diurnal cycle comparison: SPICE-aligned LST (Apollo 15)
- **Fig 4** — Per-sensor diurnal cycle comparison: SPICE-aligned LST (Apollo 17)


In [ ]:
from __future__ import annotations
import sys, pathlib, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines  import Line2D
from matplotlib.patches import Patch

from lunar.validation import load_apollo_hfe_temperature, load_apollo_hfe_depth
from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_hayne, density_hayne, specific_heat
from lunar.constants import (
    SIGMA_SB, EMISSIVITY_DEFAULT, CHI_RADIATIVE, T_REFERENCE,
    K_SURFACE, K_DEEP, H_PARAMETER, LUNATION_SECONDS,
)
from lunar.solver import PixelInputs, solve_pixel
from lunar.apollo_helpers import (
    extract_sensor_stability, print_stability_table,
    iso_to_seconds, find_stable_window, run_site_solvers,
    compute_validation_stats,
)
from lunar.plotting.style_guide import apply_style, COLORS, save_figure

apply_style()

REPO_ROOT  = str(_p)
OUT_DIR    = os.path.join(REPO_ROOT, 'output', 'figures')
os.makedirs(OUT_DIR, exist_ok=True)
print('Imports OK. Figures →', OUT_DIR)


## §1  Site + model parameters

All tuneable values in one place. Deep references in comments.

In [ ]:
# ── Physical constants ──────────────────────────────────────────────────
S0           = 1361.0          # W m⁻² — solar constant (Kopp & Lean 2011)
T_LUNAR      = LUNATION_SECONDS  # 29.530589 d synodic period [s]
DT_STEP      = 3600.0          # s — time step (1 h)
N_LUNATIONS  = 100             # spin-up lunations (≥80 for deep convergence)
SPINUP_TOL   = 0.01            # K — convergence threshold

# ── Depth grid ───────────────────────────────────────────────────────────
GRID = dict(z_max=5.0, dz0=0.002, growth=0.08)

# ── Apollo site parameters ───────────────────────────────────────────────
#    Q_b from Langseth et al. 1976 (A15) and Nagihara et al. 2018 (A17)
SITES = {
    'A15': dict(
        label='Apollo 15',  lat=26.13, lon=3.63,
        albedo=0.12, emissivity=EMISSIVITY_DEFAULT,
        Q_BASAL=0.021,      # 21 mW m⁻²  Langseth 1976
        T_MEAN_EFF=250.0,
        MIN_DEPTH_CM=80,
        y_lim=160,
        mission='a15',
    ),
    'A17': dict(
        label='Apollo 17',  lat=20.19, lon=30.77,
        albedo=0.14, emissivity=EMISSIVITY_DEFAULT,
        Q_BASAL=0.015,      # 15 mW m⁻²  Nagihara et al. 2018
        T_MEAN_EFF=255.0,
        MIN_DEPTH_CM=80,
        y_lim=240,
        mission='a17',
    ),
}

# ── Hayne (2017) model — from constants.py, confirmed vs App. A ──────────
HAYNE = dict(
    K_SURFACE=K_SURFACE,   # 7.4e-4 W m⁻¹ K⁻¹   Table 2
    K_DEEP=K_DEEP,         # 3.4e-3 W m⁻¹ K⁻¹   Table 2
    H_PARAM=H_PARAMETER,   # 0.06 m               Table 2
    CHI=CHI_RADIATIVE,     # 2.7                  Table 2
    T_REF=T_REFERENCE,     # 350 K                App. A
)

# ── Discrete 3-layer model — calibrated to Apollo HFE ───────────────────
DISCRETE = dict(
    H_LAYER1=0.07,         # m  top fluffy layer
    H_LAYER2=0.20,         # m  end of transition
    K_SOLID_SURF=1.0e-3,   # W m⁻¹ K⁻¹
    K_SOLID_DEEP=6.3e-3,   # W m⁻¹ K⁻¹
    RHO_SURF=1100.0,
    RHO_DEEP=1700.0,
    RHO_MAX=1800.0,
    RHO_EFOLD=0.5,         # m
    CHI=CHI_RADIATIVE,
    T_REF=T_REFERENCE,
)

# ── Colours ──────────────────────────────────────────────────────────────
CLR_HAYNE    = '#2471A3'   # blue   — Hayne 2017
CLR_DISC     = '#C0392B'   # red    — Discrete 3-layer
CLR_TG       = '#0B1D51'   # navy   — TG sensor
CLR_TR       = '#7A1B1B'   # maroon — TR sensor
ZONE_CLR     = '#E8DAEF'   # lavender — diurnal exclusion shading
ZONE_LINE    = '#7D3C98'   # purple   — exclusion boundary

# Build shared grid and time array once
grid   = make_geometric_grid(**GRID)
z_mid  = grid.z_mid
z_cm   = z_mid * 100.0
N_t    = int(T_LUNAR / DT_STEP) + 1
t_s    = np.linspace(0.0, T_LUNAR, N_t)

print(f'Grid: {grid.n_layers} layers, z_max = {z_mid[-1]:.2f} m')
print(f'Time: {N_t} steps / lunation')


## §2  Load Apollo HFE data

Stability windows extracted with |dT/dt| ≤ 0.08 K yr⁻¹ criterion on the tail of each sensor record.

In [ ]:
# ── Load stability windows for both sites ──────────────────────────────
hfe = {}
for tag, cfg in SITES.items():
    print(f'\n── {cfg["label"]} ──')
    bundle = extract_sensor_stability(cfg['mission'], cfg['MIN_DEPTH_CM'])
    hfe[tag] = bundle
    print_stability_table(bundle, cfg['label'], cfg['MIN_DEPTH_CM'])


## §3  Hayne (2017) model properties

**Fig 1** — Four-panel summary of the Hayne (2017) Appendix A model inputs: K(z), K(T), ρ(z), c_p(T). This confirms exact implementation before validation.

In [ ]:
# ── Fig 1: Hayne (2017) thermophysical properties vs depth ─────────────
# Reproduces the key inputs of Hayne (2017) Appendix A so the reader
# can verify the model is faithfully implemented before looking at
# validation results.

z_p   = np.linspace(0.001, 2.5, 500)   # m  — depth axis
T_250 = np.full_like(z_p, 250.0)        # K  — representative mid-T

# ── Hayne properties ────────────────────────────────────────────────────
K_h_250 = conductivity_hayne(T_250, z_p)
K_h_100 = conductivity_hayne(np.full_like(z_p, 100.0), z_p)
K_h_380 = conductivity_hayne(np.full_like(z_p, 380.0), z_p)
rho_h   = density_hayne(z_p)
cp_T    = np.linspace(50, 400, 400)
cp_h    = specific_heat(cp_T, model='hayne')
Gamma_h = np.sqrt(K_h_250 * rho_h * specific_heat(T_250))   # thermal inertia

# ── Discrete 3-layer properties ─────────────────────────────────────────
H1, H2 = DISCRETE['H_LAYER1'], DISCRETE['H_LAYER2']
Ks_d, Kd_d = DISCRETE['K_SOLID_SURF'], DISCRETE['K_SOLID_DEEP']
def _k_disc_solid(z):
    return np.where(z < H1, Ks_d,
           np.where(z < H2, Ks_d + (Kd_d - Ks_d)*(z-H1)/(H2-H1), Kd_d))
K_d_250  = _k_disc_solid(z_p) * (1 + DISCRETE['CHI'] * (250/350)**3)
rho_disc = np.where(z_p < H1, DISCRETE['RHO_SURF'],
           np.where(z_p < H2,
               DISCRETE['RHO_SURF'] + (DISCRETE['RHO_DEEP']-DISCRETE['RHO_SURF'])*(z_p-H1)/(H2-H1),
               DISCRETE['RHO_DEEP'] + (DISCRETE['RHO_MAX']-DISCRETE['RHO_DEEP'])
                   * (1-np.exp(-(z_p-H2)/DISCRETE['RHO_EFOLD']))))

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5.5), constrained_layout=True)

# (a) K(z) at 250 K — main comparison
ax = axes[0]
ax.plot(K_h_250*1e3, z_p*100, color=CLR_HAYNE, lw=2.2, ls='--', label='Hayne 2017')
ax.plot(K_d_250*1e3, z_p*100, color=CLR_DISC,  lw=2.2, ls='-',  label='Discrete')
ax.axhspan(0, 80, color='gold', alpha=0.07, zorder=0)
for h_cm, lbl in [(H1*100, 'L1→2'), (H2*100, 'L2→3')]:
    ax.axhline(h_cm, color='0.55', lw=0.8, ls=':', zorder=1)
    ax.text(ax.get_xlim()[1] if ax.get_xlim()[1]>0 else 1, h_cm-1, lbl, fontsize=7, color='0.5', va='top', ha='right')
ax.invert_yaxis(); ax.set_ylim(200, 0)
ax.set_xlabel('K(T=250 K, z)  [mW m⁻¹ K⁻¹]', fontsize=9)
ax.set_ylabel('Depth  [cm]', fontsize=9)
ax.set_title('(a)  Thermal conductivity', fontsize=10, weight='bold')
ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.2)

# (b) K temperature sensitivity at surface (z→0)
ax = axes[1]
T_arr = np.linspace(50, 400, 300)
Ks_T  = K_SURFACE * (1 + CHI_RADIATIVE * (T_arr/350)**3)
ax.plot(T_arr, Ks_T*1e3, color=CLR_HAYNE, lw=2.2)
ax.set_xlabel('Temperature  [K]', fontsize=9)
ax.set_ylabel('K(T, z=0)  [mW m⁻¹ K⁻¹]', fontsize=9)
ax.set_title('(b)  K(T) surface — radiative term', fontsize=10, weight='bold')
ax.grid(alpha=0.2)
ax.annotate('χ = 2.7\n(Hayne 2017 Table 2)', xy=(350, float(K_SURFACE*(1+2.7)*1e3)),
            fontsize=8, color=CLR_HAYNE,
            xytext=(270, float(K_SURFACE*(1+2.7)*1e3)+0.6),
            arrowprops=dict(arrowstyle='->', color=CLR_HAYNE, lw=1.2))

# (c) Density
ax = axes[2]
ax.plot(rho_h,    z_p*100, color=CLR_HAYNE, lw=2.2, ls='--', label='Hayne 2017')
ax.plot(rho_disc, z_p*100, color=CLR_DISC,  lw=2.2, ls='-',  label='Discrete')
ax.axhspan(0, 80, color='gold', alpha=0.07, zorder=0)
ax.invert_yaxis(); ax.set_ylim(200, 0)
ax.set_xlabel('Bulk density  [kg m⁻³]', fontsize=9)
ax.set_ylabel('Depth  [cm]', fontsize=9)
ax.set_title('(c)  Density profile', fontsize=10, weight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# (d) c_p(T) — Hayne App. A polynomial
ax = axes[3]
ax.plot(cp_T, cp_h, color='#117A65', lw=2.2)
ax.set_xlabel('Temperature  [K]', fontsize=9)
ax.set_ylabel('c_p  [J kg⁻¹ K⁻¹]', fontsize=9)
ax.set_title('(d)  Specific heat — Hayne App. A', fontsize=10, weight='bold')
ax.grid(alpha=0.2)

fig.suptitle('Hayne (2017) Thermophysical Properties — Model Inputs\n'
             'Equations: Hayne et al. 2017, Appendix A / Table 2  (verified)',
             fontsize=11, weight='bold')
save_figure(fig, 'hayne_model_properties', output_dir=OUT_DIR)
plt.show()


## §4  Run thermal solvers

Both models run at each site (100-lunation spin-up). Only K(T,z) and rho(z) differ.

In [ ]:
# Discrete 3-layer property functions
def k_discrete(T, z):
    H1, H2 = DISCRETE['H_LAYER1'], DISCRETE['H_LAYER2']
    Ks, Kd = DISCRETE['K_SOLID_SURF'], DISCRETE['K_SOLID_DEEP']
    chi, T_ref = DISCRETE['CHI'], DISCRETE['T_REF']
    z = np.asarray(z, dtype=float); T = np.asarray(T, dtype=float)
    ks = np.where(z < H1, Ks,
         np.where(z < H2, Ks + (Kd-Ks)*(z-H1)/(H2-H1), Kd))
    return ks * (1.0 + chi*(T/T_ref)**3)

def rho_discrete(z):
    H1, H2 = DISCRETE['H_LAYER1'], DISCRETE['H_LAYER2']
    rs, rd, rm, re = (DISCRETE['RHO_SURF'], DISCRETE['RHO_DEEP'],
                      DISCRETE['RHO_MAX'],  DISCRETE['RHO_EFOLD'])
    z = np.asarray(z, dtype=float)
    return np.where(z < H1, rs,
           np.where(z < H2, rs + (rd-rs)*(z-H1)/(H2-H1),
               rd + (rm-rd)*(1 - np.exp(-(z-H2)/re))))

print('Discrete model functions defined.')


In [ ]:
# Run Hayne + Discrete solvers for both sites
runs  = {}
stats = {}

for tag, cfg in SITES.items():
    print(f'\n---- {cfg["label"]} (lat={cfg["lat"]}N, Q_b={cfg["Q_BASAL"]*1e3:.0f} mW/m2) ----')
    run = run_site_solvers(
        cfg, grid, t_s, HAYNE,
        K_func_hayne=conductivity_hayne,
        K_func_disc=k_discrete,
        rho_func_disc=rho_discrete,
        cp_func=specific_heat,
        s0_nominal=S0, sun_scale=1.0,
        t_lunar=T_LUNAR, n_lunations=N_LUNATIONS,
        spinup_tol=SPINUP_TOL,
    )
    st = compute_validation_stats(hfe[tag], run, z_cm)
    runs[tag]  = run
    stats[tag] = st
    n_d = hfe[tag]['deep_mask'].sum()
    d   = cfg['MIN_DEPTH_CM']
    print(f'  Hayne    RMSE(z>={d}cm, N={n_d}) = {st["rmse_hayne"]:.3f} K  bias = {st["bias_hayne"]:+.3f} K')
    print(f'  Discrete RMSE(z>={d}cm, N={n_d}) = {st["rmse_disc"]:.3f} K  bias = {st["bias_disc"]:+.3f} K')

print('Both sites done.')


## §5  Mean T(z) profile validation — Fig 2

**Fig 2** shows the time-averaged temperature profile T(z) from each model overlaid on the Apollo HFE equilibrium temperatures. Sensors at z < 80 cm are shown but excluded from RMSE scoring due to the fibreglass borestem heat-short artefact (Nagihara et al. 2018).

In [ ]:
# ---- Fig 2: Mean T(z) validation, A15 + A17 side by side ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=False,
                         constrained_layout=True)

_stype_marker = {'TG': 'o', 'TR': 's'}
_stype_ms     = {'TG': 9,   'TR': 7}

for ax, (tag, cfg) in zip(axes, SITES.items()):
    bundle = hfe[tag]
    st     = stats[tag]
    label  = cfg['label']
    min_d  = cfg['MIN_DEPTH_CM']
    y_lim  = cfg['y_lim']

    # Diurnal exclusion zone
    ax.axhspan(0, min_d, color=ZONE_CLR, alpha=0.45, zorder=0)
    ax.axhline(min_d, color=ZONE_LINE, lw=1.0, ls='--', alpha=0.7, zorder=1)
    ax.text(0.98, min_d + 1, f'Excl. zone < {min_d} cm',
            fontsize=7.5, color=ZONE_LINE, va='bottom', ha='right',
            transform=ax.get_yaxis_transform(), zorder=2)

    # Model profiles
    z_model = z_cm
    ax.plot(st['T_mean_hayne'], z_model, color=CLR_HAYNE, lw=2.2,
            ls='--', label='Hayne 2017', zorder=4)
    ax.plot(st['T_mean_disc'],  z_model, color=CLR_DISC,  lw=2.2,
            ls='-',  label='Discrete 3L', zorder=5)

    # Apollo HFE data points
    for st_type in ['TG', 'TR']:
        mask = np.array([s == st_type for s in bundle['stype_all']])
        if mask.any():
            clr = CLR_TG if st_type == 'TG' else CLR_TR
            lbl = f'Apollo {st_type}'
            ax.errorbar(
                bundle['T_eq_all'][mask], bundle['depth_cm_all'][mask],
                xerr=bundle['T_std_all'][mask],
                fmt=_stype_marker[st_type], markersize=_stype_ms[st_type],
                color=clr, ecolor=clr, capsize=3, alpha=0.85,
                markeredgewidth=1.0, markeredgecolor='white',
                label=lbl, zorder=6,
            )

    # Geothermal gradient reference line
    z_deep = np.array([min_d, y_lim]) / 100.0  # m
    K_deep_val = float(conductivity_hayne(
        np.full(2, st['T_mean_hayne'][-1]), np.array([1.0, 2.0]))[0])
    T_gradient = (st['T_mean_hayne'][-1]
                  + cfg['Q_BASAL'] * (z_deep[-1] - z_deep[0]) / K_deep_val)

    # Formatting
    ax.invert_yaxis()
    ax.set_ylim(y_lim, 0)
    ax.set_xlabel('Temperature  [K]', fontsize=10)
    ax.set_ylabel('Depth  [cm]', fontsize=10)
    ax.set_title(
        f'{label}  ({cfg["lat"]}°N)\n'
        f'Hayne RMSE={st["rmse_hayne"]:.2f} K  |  '
        f'Discrete RMSE={st["rmse_disc"]:.2f} K  (z>={min_d} cm)',
        fontsize=10, weight='bold'
    )
    ax.legend(fontsize=8.5, loc='lower right')
    ax.grid(True, alpha=0.2)

fig.suptitle(
    'Apollo 15 & 17 HFE — Annual-Mean Temperature Profile\n'
    'Hayne (2017) vs Discrete 3-layer model vs In-situ observations',
    fontsize=12, weight='bold'
)
save_figure(fig, 'apollo_mean_T_profile', output_dir=OUT_DIR)
plt.show()
print('Saved: apollo_mean_T_profile.{png,pdf}')


## §6  Per-sensor diurnal cycles — Figs 3 & 4

Local solar time is computed directly from SPICE (MOON_ME frame subsolar longitude), giving a deterministic phase reference with ~4-min accuracy (Nagihara et al. 2018). Phase-folded Apollo medians (20-min LST bins) are overlaid on both models.

**Panel backgrounds:** green = validation zone (z ≥ 80 cm, used in RMSE); red = borestem-contaminated zone (z < 80 cm, excluded). The shallow sensors are **shown** so the reader can see the borestem artefact — they are NOT used for any quantitative metric.

**Phase shift correction:** solid/dashed bright lines are peak-aligned (model peak shifted to match observed peak) to isolate amplitude errors from any phase offset. Dim ghost lines show unshifted model for reference.

In [ ]:
# Build SPICE LST lookup table (once, shared by both sites)
from lunar.ephem import _furnish_kernels as _furnish_spice
import spiceypy as _spice
from datetime import datetime, timezone

_furnish_spice()

T_SYN_HR = 29.530589 * 24.0   # 708.734 h

def _iso_to_unix(s):
    s2 = s.rstrip('Z') + '+00:00' if s.endswith('Z') else s
    return datetime.fromisoformat(s2).replace(tzinfo=timezone.utc).timestamp()

def _unix_to_et(arr):
    return np.array([_spice.unitim(u/86400.0 + 2440587.5, 'JED', 'ET')
                     for u in arr], dtype=np.float64)

_N_GRID    = int(42 * 365.25)
_unix_grid = np.linspace(_iso_to_unix('1970-01-01T00:00:00'),
                         _iso_to_unix('2012-01-01T00:00:00'), _N_GRID)
_et_grid   = _unix_to_et(_unix_grid)

_sub_lon_raw = np.empty(_N_GRID)
for _i, _et in enumerate(_et_grid):
    _pos, _ = _spice.spkpos('SUN', float(_et), 'MOON_ME', 'LT+S', 'MOON')
    _sub_lon_raw[_i] = np.rad2deg(np.arctan2(_pos[1], _pos[0]))
_sub_lon_unwrap = np.unwrap(np.deg2rad(_sub_lon_raw))

def spice_lst(t_unix, site_lon):
    t    = np.atleast_1d(np.asarray(t_unix, dtype=float))
    slon = np.rad2deg(np.interp(t, _unix_grid, _sub_lon_unwrap)) % 360.0
    HA   = ((site_lon - slon + 180.0) % 360.0) - 180.0
    return (12.0 + HA / 15.0) % 24.0

def lst_to_lunhour(lst_hr):
    return np.asarray(lst_hr, float) * (T_SYN_HR / 24.0)

def _bin_med_iqr(lst, T, edges):
    nb = len(edges) - 1
    med = np.full(nb, np.nan); q25 = med.copy(); q75 = med.copy()
    for k in range(nb):
        sel = (lst >= edges[k]) & (lst < edges[k+1])
        if sel.sum() >= 3:
            med[k] = np.median(T[sel])
            q25[k] = np.percentile(T[sel], 25)
            q75[k] = np.percentile(T[sel], 75)
    return med, q25, q75

def _pi(lst_q, lst_m, T_m):
    x = np.asarray(lst_m, float) % 24.0
    y = np.asarray(T_m, float)
    s = np.argsort(x)
    x2 = np.r_[x[s], x[s]+24.0]; y2 = np.r_[y[s], y[s]]
    return np.interp(np.asarray(lst_q, float) % 24.0, x2, y2)

N_BINS    = 72
BIN_EDGES = np.linspace(0, 24, N_BINS + 1)
BIN_CTR   = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
BIN_LUNHR = lst_to_lunhour(BIN_CTR)

# Model LST mapping (t=0 = local noon for sinusoidal forcing)
_t_int    = t_s[:-1]
LST_MOD   = ((_t_int - T_LUNAR/2.0) % T_LUNAR) / T_LUNAR * 24.0

H_SUNRISE = T_SYN_HR / 4.0
H_NOON    = T_SYN_HR / 2.0
H_SUNSET  = 3.0 * T_SYN_HR / 4.0

print(f'SPICE LST table: {_N_GRID} daily points (1970-2012)')


In [ ]:
# Figs 3 & 4: per-sensor diurnal grid for A15 and A17
# -------------------------------------------------------
# Colours for diurnal panels (distinct from profile panels)
CLR_H_D = '#E67E22'   # orange  — Hayne diurnal
CLR_D_D = '#1565C0'   # blue    — Discrete diurnal
ZONE_EXCL = '#FADBD8'; ZONE_VAL = '#D4EFDF'; NIGHT_G = '#ececec'

def _diurnal_site_figure(tag, figname):
    cfg     = SITES[tag]
    bundle  = hfe[tag]
    out_h   = runs[tag]['out_hayne']
    out_d   = runs[tag]['out_disc']
    site_lon = cfg['lon']
    min_d    = cfg['MIN_DEPTH_CM']

    # Phase-fold Apollo observations with SPICE LST
    obs_all = {}
    for dtab in [bundle['d1'], bundle['d2']]:
        for sensor in np.unique(dtab['sensor']):
            mask  = dtab['sensor'] == sensor
            t_u   = iso_to_seconds(dtab['time_iso'][mask])
            T_obs = dtab['T'][mask].astype(float)
            good  = np.isfinite(T_obs) & (T_obs > 50.0)
            t_u, T_obs = t_u[good], T_obs[good]
            if len(t_u) < 10:
                continue
            lst = spice_lst(t_u, site_lon)
            key = sensor.strip()
            nh = len(lst) // 2   # use second half (skip drilling transient)
            obs_all[key] = (lst[nh:], T_obs[nh:])

    sorted_sens = sorted(bundle['sensors'], key=lambda s: s['depth_cm'])
    n_s = len(sorted_sens)
    NCOLS = 3
    NROWS = int(np.ceil(n_s / NCOLS))

    fig, axes = plt.subplots(NROWS, NCOLS,
                             figsize=(5.2*NCOLS, 3.0*NROWS),
                             constrained_layout=False)
    axes_flat = np.array(axes).reshape(-1)

    for idx, s in enumerate(sorted_sens):
        ax       = axes_flat[idx]
        sensor   = s['sensor']
        depth_cm = s['depth_cm']
        is_deep  = depth_cm >= min_d
        iz       = int(np.argmin(np.abs(z_cm - depth_cm)))

        ax.set_facecolor(ZONE_VAL if is_deep else ZONE_EXCL)
        ax.axvspan(0.0, H_SUNRISE, color=NIGHT_G, alpha=0.6, zorder=0)
        ax.axvspan(H_SUNSET, T_SYN_HR, color=NIGHT_G, alpha=0.6, zorder=0)
        for xv in (H_SUNRISE, H_NOON, H_SUNSET):
            ax.axvline(xv, color='#F39C12', ls=':', lw=0.7, alpha=0.6, zorder=1)

        # Model curves at this depth — unshifted and peak-aligned
        _Th = out_h.T[iz, :-1];  _Td = out_d.T[iz, :-1]
        mod_pk_h = float(LST_MOD[np.argmax(_Th)])
        mod_pk_d = float(LST_MOD[np.argmax(_Td)])

        _obs_pk = np.nan
        if sensor in obs_all:
            _lo, _To = obs_all[sensor]
            _mt, _, _ = _bin_med_iqr(_lo, _To, BIN_EDGES)
            _ok = np.isfinite(_mt)
            if _ok.sum() >= 5:
                _obs_pk = float(BIN_CTR[_ok][np.argmax(_mt[_ok])])

        if np.isfinite(_obs_pk):
            lag_h = (mod_pk_h - _obs_pk + 12.0) % 24.0 - 12.0
            lag_d = (mod_pk_d - _obs_pk + 12.0) % 24.0 - 12.0
        else:
            lag_h = lag_d = 0.0

        T_h_u = _pi(BIN_CTR, LST_MOD, _Th)
        T_d_u = _pi(BIN_CTR, LST_MOD, _Td)
        T_h_s = _pi((BIN_CTR + lag_h) % 24.0, LST_MOD, _Th)
        T_d_s = _pi((BIN_CTR + lag_d) % 24.0, LST_MOD, _Td)

        ax.plot(BIN_LUNHR, T_h_u, color=CLR_H_D, lw=1.0, ls='--', alpha=0.30, zorder=3)
        ax.plot(BIN_LUNHR, T_d_u, color=CLR_D_D, lw=1.0, ls='-',  alpha=0.30, zorder=3)
        ax.plot(BIN_LUNHR, T_h_s, color=CLR_H_D, lw=1.8, ls='--', alpha=1.00, zorder=4)
        ax.plot(BIN_LUNHR, T_d_s, color=CLR_D_D, lw=2.2, ls='-',  alpha=1.00, zorder=5)

        # Apollo observations
        if sensor in obs_all:
            _lo, _To = obs_all[sensor]
            med, q25, q75 = _bin_med_iqr(_lo, _To, BIN_EDGES)
            ok = np.isfinite(med)
            if ok.any():
                stype = s.get('stype', sensor[:2])
                clr_o = CLR_TG if stype == 'TG' else CLR_TR
                mrk   = 'o'    if stype == 'TG' else 's'
                ax.fill_between(BIN_LUNHR[ok], q25[ok], q75[ok],
                                color=clr_o, alpha=0.10, zorder=2)
                ax.plot(BIN_LUNHR[ok], med[ok], marker=mrk, markersize=3,
                        ls='none', color=clr_o, zorder=6)

        lbl_tag = '[valid.]' if is_deep else '[borestem]'
        t_clr   = '#186A3B' if is_deep else '#A93226'
        lag_str = f'  dH={lag_h:+.1f}h dD={lag_d:+.1f}h' if np.isfinite(_obs_pk) else ''
        ax.set_title(f'{sensor}  z={depth_cm:.0f} cm  {lbl_tag}{lag_str}',
                     fontsize=8.5, weight='bold', color=t_clr, pad=3)
        ax.set_xlim(0, T_SYN_HR)
        ax.set_xticks([0, H_SUNRISE, H_NOON, H_SUNSET, T_SYN_HR])
        ax.set_xticklabels(['0h', f'{H_SUNRISE:.0f}h',
                            f'{H_NOON:.0f}h', f'{H_SUNSET:.0f}h',
                            f'{T_SYN_HR:.0f}h'], fontsize=7.5)
        ax.tick_params(axis='y', labelsize=8.5)
        if idx % NCOLS == 0:
            ax.set_ylabel('T  [K]', fontsize=9)
        if idx // NCOLS == NROWS - 1:
            ax.set_xlabel('Elapsed lunar hours (midnight=0)', fontsize=9)

        # Tight y limits
        _lo_y = min(T_h_u.min(), T_d_u.min(), T_h_s.min(), T_d_s.min())
        _hi_y = max(T_h_u.max(), T_d_u.max(), T_h_s.max(), T_d_s.max())
        if sensor in obs_all:
            _lo_obs, _To_obs = obs_all[sensor]
            _lo_y = min(_lo_y, float(np.percentile(_To_obs, 1)))
            _hi_y = max(_hi_y, float(np.percentile(_To_obs, 99)))
        pad = 0.15 * (_hi_y - _lo_y + 0.2)
        ax.set_ylim(_lo_y - pad, _hi_y + pad)

    for k in range(n_s, len(axes_flat)):
        axes_flat[k].set_visible(False)

    # Shared legend
    handles = [
        Line2D([], [], marker='o', ls='none', color=CLR_TG, label='Apollo TG (median ± IQR)'),
        Line2D([], [], marker='s', ls='none', color=CLR_TR, label='Apollo TR (median ± IQR)'),
        Line2D([], [], color=CLR_H_D, lw=1.8, ls='--', label='Hayne 2017 (peak-aligned)'),
        Line2D([], [], color=CLR_D_D, lw=2.2, ls='-',  label='Discrete 3L (peak-aligned)'),
        Line2D([], [], color=CLR_H_D, lw=1.0, ls='--', alpha=0.35, label='Hayne (unshifted)'),
        Line2D([], [], color=CLR_D_D, lw=1.0, ls='-',  alpha=0.35, label='Discrete (unshifted)'),
        Patch(facecolor=ZONE_VAL,  edgecolor='#186A3B', label=f'Validation (z>={min_d} cm)'),
        Patch(facecolor=ZONE_EXCL, edgecolor='#A93226', label='Borestem zone (excluded)'),
    ]
    fig.legend(handles=handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.01), ncol=4, fontsize=8.8,
               frameon=True, handlelength=2.0, columnspacing=1.5)
    fig.suptitle(
        f'{cfg["label"]} HFE — Diurnal Cycle Comparison (SPICE LST)\n'
        f'Peak-aligned model vs Apollo binned medians',
        fontsize=11.5, weight='bold'
    )
    fig.subplots_adjust(top=0.93, bottom=0.13, left=0.055, right=0.985,
                        hspace=0.52, wspace=0.22)
    save_figure(fig, figname, output_dir=OUT_DIR)
    plt.show()
    print(f'Saved: {figname}')

# Generate both sites in one cell
_diurnal_site_figure('A15', 'a15_diurnal_sensor_grid')
_diurnal_site_figure('A17', 'a17_diurnal_sensor_grid')


## §7  Validation statistics

RMSE, bias, MAE, and R² for the deep sensors (z ≥ 80 cm) at both sites. Phase-1 success criterion: RMSE ≤ 2 K at depth for a single-pixel validation.

In [ ]:
from scipy.stats import pearsonr

print('=' * 75)
print('APOLLO VALIDATION — DEEP-SENSOR STATISTICS  (z >= MIN_DEPTH_CM)')
print('=' * 75)

for tag, cfg in SITES.items():
    bundle = hfe[tag]
    st     = stats[tag]
    min_d  = cfg['MIN_DEPTH_CM']
    n_deep = bundle['deep_mask'].sum()

    print(f'\n{cfg["label"]}  (z >= {min_d} cm,  N = {n_deep})')
    print(f'  {"":30s}  {"Hayne 2017":>14}  {"Discrete 3L":>13}')
    print(f'  {"─"*60}')
    for label, h_val, d_val in [
        ('RMSE  [K]',      st['rmse_hayne'],       st['rmse_disc']),
        ('Bias  [K]',      st['bias_hayne'],       st['bias_disc']),
        ('MAE   [K]',      st['mae_hayne'],        st['mae_disc']),
        ('R²',             st['r2_hayne'],         st['r2_disc']),
    ]:
        fmt = '+.3f' if 'Bias' in label else '.4f'
        print(f'  {label:30s}  {h_val:{fmt}:>14}  {d_val:{fmt}:>13}')

    # Per-sensor table
    print(f'\n  Per-sensor residuals  (model_mean - T_eq):')
    print(f'  {"Sensor":<8} {"z[cm]":>6} {"T_obs":>8} {"Hayne":>8} {"Disc":>8} {"dH":>7} {"dD":>7} {"valid"}')
    print(f'  {"─"*70}')
    for i, s in enumerate(bundle['sensors']):
        mark = 'Y' if bundle['deep_mask'][i] else '-'
        T_h = float(np.interp(s['depth_cm'], z_cm, st['T_mean_hayne']))
        T_d = float(np.interp(s['depth_cm'], z_cm, st['T_mean_disc']))
        dh  = T_h - s['T_eq']
        dd  = T_d - s['T_eq']
        print(f'  {s["sensor"]:<8} {s["depth_cm"]:>6.0f} {s["T_eq"]:>8.2f} '
              f'{T_h:>8.2f} {T_d:>8.2f} {dh:>+7.2f} {dd:>+7.2f}  {mark}')

print('\nPhase-1 success criterion: RMSE <= 2 K at depth  -> ',
      all(stats[t]['rmse_hayne'] <= 2.0 for t in SITES) and
      all(stats[t]['rmse_disc']  <= 2.0 for t in SITES))


## §8  Borestem heat-short artefact — why shallow sensors are excluded

At z < 80 cm, both Apollo missions have sensors inside the fibreglass borestem whose axial thermal conductance (~0.25 W m⁻¹ K⁻¹) short-circuits the regolith-diffusion signal. The diurnal skin depth at Apollo 15 is δ ≈ 3-5 cm, so z = 35 cm sits 7-10 skin depths deep; pure regolith diffusion gives ~0.6-1 K amplitude there. The Apollo record shows ~5.5 K — the excess is the borestem conducting surface heat down the probe.

This is **not a model defect** — it is a well-documented hardware artefact (Langseth et al. 1977; Grott et al. 2010; Nagihara et al. 2018 §3.2). Attempting to tune the model to match the shallow sensors would require breaking Hayne (2017) constants and would invalidate the deep-sensor validation. The correct approach (used here) is to show the artefact transparently and score only the deep sensors.

### Amplitude table (expected vs observed)

| Depth [cm] | Apollo amp [K] | Hayne model [K] | Expected ratio |
|---|---|---|---|
| 35 | ~5.5 | ~0.6 | e⁻⁷ ≈ 0.1% — artefact dominates |
| 84–129 | 0.04–0.14 | 0.01 | Both sub-Kelvin — consistent |

**References:** Nagihara et al. 2018 §3.2; Grott et al. 2010 JGR Planets 115 E11005.